# 05 – Support Vector Machine
**CS 5331/7331 · Project Team Five**

Trains an SVM classifier on both feature sets (full 30 / no-external 25), with and without SMOTEN.
Features are standardized inside the pipeline before fitting. Hyperparameters are tuned with 5-fold stratified CV on the training set (scored on F1), then each best model is scored once on the test set.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imbalanced-learn

import os, json
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTEN

BASE = '/content/drive/MyDrive/CS7331_Phishing'
DATA_DIR = f'{BASE}/data'
RESULTS_DIR = f'{BASE}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
SEED = 42

# load the exact splits Safia saved from preprocessing
X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')['phishing']
y_test = pd.read_csv(f'{DATA_DIR}/y_test.csv')['phishing']
with open(f'{DATA_DIR}/feature_sets.json') as f:
    FEATURE_SETS = json.load(f)

print(X_train.shape, X_test.shape, y_train.mean().round(3))

Mounted at /content/drive
(4679, 30) (1170, 30) 0.516


In [2]:
from joblib import Memory
CACHE = Memory('/content/smote_cache', verbose=0)  # remembers SMOTEN results so it doesn't redo the same thing
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# SVM hyperparams:
#  - kernel picks the decision boundary shape (rbf = flexible non-linear, linear = straight hyperplane)
#  - C controls how much to penalize misclassified points (higher = tighter fit, more overfitting risk)
#  - gamma for rbf controls how far one training point's influence reaches ('scale' is 1 / (n_features * X.var()))
# linear kernel doesn't use gamma so it gets its own sub-grid
GRID = [
    {'svm__kernel': ['rbf'], 'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 0.1]},
    {'svm__kernel': ['linear'], 'svm__C': [0.1, 1, 10]},
]

def make_pipe(use_smote):
    steps = []
    if use_smote:
        steps.append(('smote', SMOTEN(random_state=SEED)))  # inside the pipeline so it only touches training folds, test set stays untouched
    steps.append(('scaler', StandardScaler()))  # SVM needs features on the same scale
    steps.append(('svm', SVC(probability=True, random_state=SEED)))  # probability=True so we can get roc_auc
    return Pipeline(steps, memory=CACHE)

In [3]:
rows = []
preds = pd.DataFrame({'y_true': y_test.values})

for fs_name, cols in FEATURE_SETS.items():
    for use_smote in [False, True]:
        search = GridSearchCV(make_pipe(use_smote), GRID, scoring='f1', cv=cv, n_jobs=-1)
        search.fit(X_train[cols], y_train)
        print(f"done: SVM | {fs_name} | {'smote' if use_smote else 'no_smote'}")

        best = search.best_estimator_  # refit on the full training set with the best setting
        y_pred = best.predict(X_test[cols])
        y_prob = best.predict_proba(X_test[cols])[:, 1]

        smote_tag = 'smote' if use_smote else 'no_smote'
        rows.append({
            'model': 'SVM', 'features': fs_name, 'smote': smote_tag,
            'best_params': search.best_params_,
            'cv_f1': round(search.best_score_, 4),
            'accuracy': round(accuracy_score(y_test, y_pred), 4),
            'precision': round(precision_score(y_test, y_pred), 4),
            'recall': round(recall_score(y_test, y_pred), 4),
            'f1': round(f1_score(y_test, y_pred), 4),
            'roc_auc': round(roc_auc_score(y_test, y_prob), 4),
        })
        # keep predictions for the confusion matrices and ROC curves later
        key = f'SVM|{fs_name}|{smote_tag}'
        preds[f'{key}|pred'] = y_pred
        preds[f'{key}|prob'] = y_prob

results = pd.DataFrame(rows)
results.sort_values('cv_f1', ascending=False)

done: SVM | full | no_smote
done: SVM | full | smote
done: SVM | no_external | no_smote
done: SVM | no_external | smote


,model,features,smote,best_params,cv_f1,accuracy,precision,recall,f1,roc_auc
1,SVM,full,smote,"{'svm__C': 1, 'svm__gamma': 0.1, 'svm__kernel'...",0.9494,0.9453,0.9561,0.9371,0.9465,0.9762
0,SVM,full,no_smote,"{'svm__C': 1, 'svm__gamma': 0.1, 'svm__kernel'...",0.9487,0.9436,0.9529,0.9371,0.9449,0.9773
2,SVM,no_external,no_smote,"{'svm__C': 10, 'svm__gamma': 'scale', 'svm__ke...",0.9446,0.9410,0.9620,0.9222,0.9417,0.9814
3,SVM,no_external,smote,"{'svm__C': 10, 'svm__gamma': 'scale', 'svm__ke...",0.9436,0.9393,0.9619,0.9189,0.9399,0.9811


In [4]:
results.to_csv(f'{RESULTS_DIR}/svm_results.csv', index=False)
preds.to_csv(f'{RESULTS_DIR}/svm_predictions.csv', index=False)
print("Saved svm_results.csv and svm_predictions.csv to /results")

Saved svm_results.csv and svm_predictions.csv to /results
